In [2]:
!apt-get install osmium-tool

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libboost-program-options1.74.0
The following NEW packages will be installed:
  libboost-program-options1.74.0 osmium-tool
0 upgraded, 2 newly installed, 0 to remove and 2 not upgraded.
Need to get 882 kB of archives.
After this operation, 3,863 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-program-options1.74.0 amd64 1.74.0-14ubuntu3 [311 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 osmium-tool amd64 1.14.0-1 [571 kB]
Fetched 882 kB in 0s (3,375 kB/s)
Selecting previously unselected package libboost-program-options1.74.0:amd64.
(Reading database ... 118242 files and directories currently installed.)
Preparing to unpack .../libboost-program-options1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-program-options1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selec

In [3]:
!osmium tags-filter sri-lanka-260529.osm.pbf \
nwr/amenity=school,university,college,hospital,clinic,bus_station,marketplace,place_of_worship \
nwr/highway=bus_stop \
nwr/railway=station,halt \
nwr/tourism=attraction,museum,hotel,guest_house \
nwr/building=apartments \
nwr/landuse=industrial \
-o filtered_pois.osm.pbf --overwrite

[======================================================================] 100% 


In [1]:
import pandas as pd
import geopandas as gpd
from pyrosm import OSM
from scipy.spatial import cKDTree
import numpy as np
import gc
import os

def ultimate_ram_safe_silver_layer(pbf_file_path, csv_file_path, output_csv_path, radius=1000):
    print("1. Loading Pre-Filtered OSM Data (Strict Mode)...")
    osm = OSM(pbf_file_path)

    # STRICT FILTER: We only want these exact things, nothing else.
    custom_filter = {
        'amenity': ['school', 'university', 'college', 'hospital', 'clinic', 'bus_station', 'marketplace', 'place_of_worship'],
        'highway': ['bus_stop'],
        'railway': ['station', 'halt'],
        'tourism': ['attraction', 'museum', 'hotel', 'guest_house'],
        'building': ['apartments'],
        'landuse': ['industrial']
    }

    pois = osm.get_data_by_custom_criteria(custom_filter=custom_filter, keep_nodes=True, keep_ways=True, keep_relations=False)

    print(f" -> Successfully loaded {len(pois)} POIs.")
    if len(pois) > 100000:
        print(" ⚠️ WARNING: Your POI count is massively high! Make sure you are using the tiny 'filtered_pois.osm.pbf' file, NOT the full country map.")

    def extract_poi_type(row):
        for col in ['amenity', 'highway', 'railway', 'tourism', 'building', 'landuse']:
            if col in pois.columns and pd.notna(row[col]):
                return f"{row[col]}"
        return "unknown"

    pois['POI_Type'] = pois.apply(extract_poi_type, axis=1)
    pois['POI_ID'] = [f"POI_{str(i).zfill(6)}" for i in range(len(pois))]

    pois = pois.dropna(subset=['geometry']).copy()
    pois['geometry'] = pois['geometry'].centroid
    pois_gdf = pois.to_crs("EPSG:32644")

    base_footfall_mapping = {
        'station': 25000, 'bus_station': 10000, 'hospital': 5000,
        'university': 4000, 'marketplace': 3000, 'college': 2000,
        'school': 1000, 'apartments': 500, 'place_of_worship': 300,
        'clinic': 200, 'bus_stop': 150, 'hotel': 100,
        'guest_house': 50, 'industrial': 1000
    }
    pois_gdf['Base_Footfall'] = pois_gdf['POI_Type'].map(base_footfall_mapping).fillna(100)

    # Create lean Numpy arrays
    poi_coords = np.array(list(zip(pois_gdf.geometry.x, pois_gdf.geometry.y)))
    poi_ids = pois_gdf['POI_ID'].values
    poi_types = pois_gdf['POI_Type'].values
    poi_footfalls = pois_gdf['Base_Footfall'].values

    # DESTROY the heavy Geopandas and Pyrosm objects completely
    del osm, pois, pois_gdf
    gc.collect()

    print("2. Building KD-Tree...")
    tree = cKDTree(poi_coords)

    print("3. Loading Outlets...")
    outlets_df = pd.read_csv(csv_file_path)
    outlets_gdf = gpd.GeoDataFrame(
        outlets_df, geometry=gpd.points_from_xy(outlets_df['Longitude'], outlets_df['Latitude']), crs="EPSG:4326"
    ).to_crs("EPSG:32644")

    outlet_coords = np.array(list(zip(outlets_gdf.geometry.x, outlets_gdf.geometry.y)))
    outlet_ids = outlets_df['Outlet_ID'].values

    del outlets_df, outlets_gdf
    gc.collect()

    if os.path.exists(output_csv_path):
        os.remove(output_csv_path)

    total_outlets = len(outlet_ids)

    # ---------------------------------------------------------
    # MICRO-CHUNKING: Just 100 outlets at a time!
    # ---------------------------------------------------------
    chunk_size = 100
    sigma = 300
    total_rows = 0

    print(f"4. Calculating connections in micro-chunks of {chunk_size}...")

    for start_idx in range(0, total_outlets, chunk_size):
        end_idx = min(start_idx + chunk_size, total_outlets)

        chunk_outlets_coords = outlet_coords[start_idx:end_idx]
        chunk_outlet_ids = outlet_ids[start_idx:end_idx]

        poi_indices_list = tree.query_ball_point(chunk_outlets_coords, r=radius)

        chunk_data = []

        for i, nearby_poi_indices in enumerate(poi_indices_list):
            if len(nearby_poi_indices) == 0:
                continue

            out_id = chunk_outlet_ids[i]
            out_pt = chunk_outlets_coords[i]

            nearby_coords = poi_coords[nearby_poi_indices]
            n_ids = poi_ids[nearby_poi_indices]
            n_types = poi_types[nearby_poi_indices]
            n_footfalls = poi_footfalls[nearby_poi_indices]

            distances = np.linalg.norm(nearby_coords - out_pt, axis=1)
            weights = np.exp(-(distances**2) / (2 * sigma**2))
            final_impacts = n_footfalls * weights

            for j in range(len(nearby_poi_indices)):
                chunk_data.append({
                    'Outlet_ID': out_id,
                    'POI_ID': n_ids[j],
                    'POI_Type': n_types[j],
                    'Distance_Meters': round(distances[j], 2),
                    'Base_Footfall': n_footfalls[j],
                    'Decay_Weight': round(weights[j], 4),
                    'Footfall_Impact_Score': round(final_impacts[j], 2)
                })

        if len(chunk_data) > 0:
            chunk_df = pd.DataFrame(chunk_data)
            write_header = not os.path.exists(output_csv_path)
            chunk_df.to_csv(output_csv_path, mode='a', header=write_header, index=False)
            total_rows += len(chunk_df)

            del chunk_df

        print(f" -> Outlets {end_idx}/{total_outlets} processed... Edge rows saved so far: {total_rows}")

        # AGGRESSIVE RAM FLUSH
        del chunk_data, chunk_outlets_coords, chunk_outlet_ids, poi_indices_list
        gc.collect()

    print(f"\n✅ SUCCESS! Total connections saved: {total_rows}")
    print(f"File ready for Lakehouse Silver Layer: {output_csv_path}")

# --- RUN THE SCRIPT ---
ultimate_ram_safe_silver_layer(
    pbf_file_path='filtered_pois.osm.pbf',
    csv_file_path='cleaned_outlet_coordinates.csv',
    output_csv_path='silver_layer_poi_edges.csv'
)

1. Loading Pre-Filtered OSM Data (Strict Mode)...
 -> Successfully loaded 26044 POIs.


/tmp/ipykernel_11237/1837126515.py:39: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois['geometry'] = pois['geometry'].centroid


2. Building KD-Tree...
3. Loading Outlets...
4. Calculating connections in micro-chunks of 100...
 -> Outlets 100/19960 processed... Edge rows saved so far: 758
 -> Outlets 200/19960 processed... Edge rows saved so far: 1474
 -> Outlets 300/19960 processed... Edge rows saved so far: 2064
 -> Outlets 400/19960 processed... Edge rows saved so far: 3001
 -> Outlets 500/19960 processed... Edge rows saved so far: 3679
 -> Outlets 600/19960 processed... Edge rows saved so far: 4472
 -> Outlets 700/19960 processed... Edge rows saved so far: 5458
 -> Outlets 800/19960 processed... Edge rows saved so far: 6141
 -> Outlets 900/19960 processed... Edge rows saved so far: 7071
 -> Outlets 1000/19960 processed... Edge rows saved so far: 8042
 -> Outlets 1100/19960 processed... Edge rows saved so far: 8926
 -> Outlets 1200/19960 processed... Edge rows saved so far: 9461
 -> Outlets 1300/19960 processed... Edge rows saved so far: 10333
 -> Outlets 1400/19960 processed... Edge rows saved so far: 11074
